<a href="https://colab.research.google.com/github/kkannan18/GenAI/blob/main/GenAIPrinciples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Midterm Project: Gen AI principles

In [ ]:
#Webscraping

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

MAIN_URL = "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/"  # Replace with actual URL

def get_sublinks(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    # Adjust selector to match the program's navigation structure
    links = [urljoin(url, a['href']) for a in soup.select('a[href]') if 'applied-data-science' in a['href']]
    return list(set(links))

def extract_text(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    # Adjust selectors to extract main content
    paragraphs = soup.find_all(['p', 'li'])
    return "\n".join(p.get_text(strip=True) for p in paragraphs)

# Crawl main page and sublinks
all_urls = [MAIN_URL] + get_sublinks(MAIN_URL)
section_texts = {}
for url in all_urls:
    section_texts[url] = extract_text(url)

In [ ]:
#Split the data into chunks

def chunk_text(text, max_length=500):
    sentences = text.split('. ')
    chunks, chunk = [], ""
    for sentence in sentences:
        if len(chunk) + len(sentence) < max_length:
            chunk += sentence + ". "
        else:
            chunks.append(chunk.strip())
            chunk = sentence + ". "
    if chunk:
        chunks.append(chunk.strip())
    return chunks

all_chunks = []
for url, text in section_texts.items():
    for chunk in chunk_text(text):
        all_chunks.append({"source": url, "content": chunk})

In [ ]:
#Embedding generation and vector store setup
!pip install faiss-cpu
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode([c['content'] for c in all_chunks])

# Store embeddings in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Save mapping from vector index to chunk
chunk_id_to_meta = {i: all_chunks[i] for i in range(len(all_chunks))}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 58.7 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#RAG Pipeline

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode([c['content'] for c in all_chunks])

# Store embeddings in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Save mapping from vector index to chunk
chunk_id_to_meta = {i: all_chunks[i] for i in range(len(all_chunks))}

In [ ]:
#Responsible AI Practices

def answer_question_safe(query):
    context_chunks = retrieve(query)
    if not context_chunks:
        return "I don't know."
    context = "\n".join(context_chunks)
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
    return qa_pipeline(prompt, max_length=200)[0]['generated_text']

In [ ]:
#UX Generation
!pip install streamlit
import streamlit as st

st.title("MS in Applied Data Science Q&A")
user_query = st.text_input("Ask a question about the program:")

if user_query:
    response = answer_question_safe(user_query)
    st.write("**Answer:**", response)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.0 MB/s eta 0:00:00


2025-05-03 14:25:32.893 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-03 14:25:33.048 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-05-03 14:25:33.049 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-03 14:25:33.051 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-03 14:25:33.052 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-03 14:25:33.053 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-03 14:25:33.054 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-03 14:25:33.055 Session state does not 